# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


We isolate our contracted features and explicitly engineer the staleness_position_interaction term to directly test our core research hypothesis. Missing values are filled with 0, though our selected core telemetry signals are dense.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define the label
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
y = df['is_declining_label']

# Isolate contracted base features
base_features = ['days_since_last_update', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr']
X = df[base_features].copy()

# Feature Engineering: Freshness-Decay Interaction Term
# Multiplying staleness by average position penalty to capture combined risk
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']

# Handle any potential missing values to ensure matrix is fit for ML
X = X.fillna(0)

print(f"Feature matrix X built with shape: {X.shape}")
display(X.head(3))


Feature matrix X built with shape: (30000, 6)


,days_since_last_update,content_age_days,impressions_90d,avg_position,ctr,staleness_pos_interaction
0,20,187,3803,10.6,0.76,212.0
1,25,445,15320,20.3,0.05,507.5
2,20,141,12581,36.5,0.09,730.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*   **`days_since_last_update`:** Numeric. Days elapsed since editorial modification. Available *before* the prediction window.
*   **`content_age_days`:** Numeric. Total lifespan of the URL. Available *before* the prediction window.
*   **`impressions_90d`:** Numeric. Historical search visibility in the 90-day lookback window. Available *before* prediction.
*   **`avg_position`:** Numeric. Historical SERP ranking. Available *before* prediction.
*   **`ctr`:** Numeric. Historical click-through rate. Available *before* prediction.
*   **`staleness_pos_interaction`:** Numeric (Engineered). Product of staleness and position. Derived purely from pre-window signals.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature Matrix Data Types and Null Count Check:")
print(X.info())


Feature Matrix Data Types and Null Count Check:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   days_since_last_update     30000 non-null  int64  
 1   content_age_days           30000 non-null  int64  
 2   impressions_90d            30000 non-null  int64  
 3   avg_position               30000 non-null  float64
 4   ctr                        30000 non-null  float64
 5   staleness_pos_interaction  30000 non-null  float64
dtypes: float64(3), int64(3)
memory usage: 1.4 MB
None


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
*To aggressively test for target leakage, I am fitting a shallow Decision Tree on the feature vector. If the precision score hits ~1.000, it means a feature is secretly derived from the label. A healthy, non-leaky feature set will yield realistic, imperfect precision.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import precision_score

# Fit a shallow tree to hunt for "too good to be true" leakage
leak_hunter = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
leak_hunter.fit(X, y)

preds = leak_hunter.predict(X)
precision = precision_score(y, preds)

print(f"Leakage Hunt Precision: {precision:.3f}")
if precision > 0.95:
    print("\n⚠️ WARNING: Precision is suspiciously high. Leakage detected! Check tree splits.")
else:
    print("\n✅ PASS: Precision is realistic. No obvious target leakage detected.")

print("\nTree Structure (What it learned):")
print(export_text(leak_hunter, feature_names=list(X.columns)))

Leakage Hunt Precision: 0.651

✅ PASS: Precision is realistic. No obvious target leakage detected.

Tree Structure (What it learned):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



## 4. What I excluded and why

*   **`trend_pct`:** Excluded for direct Target Leakage. It is the literal post-window percentage change used to determine if a page is declining.
*   **`trend_direction`:** Excluded for Target Leakage. It is the label itself.
*   **`search_volume`:** Excluded for Irrelevance. Prior discovery proved it has a near-zero correlation with actual traffic outcomes in this dataset.
*   **`word_count`:** Excluded for Sparsity. ~25% of rows are missing this value; imputing a quarter of the dataset introduces unnecessary bias when high-fidelity Search Console signals are available.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = ['trend_pct', 'trend_direction', 'search_volume', 'word_count']
print(f"Fields intentionally excluded from X to guarantee model honesty: {excluded_fields}")

Fields intentionally excluded from X to guarantee model honesty: ['trend_pct', 'trend_direction', 'search_volume', 'word_count']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.